# Training OCR Aksara Jawa — EfficientNetV2-B0

Notebook ini melatih model klasifikasi aksara Jawa menggunakan **EfficientNetV2-B0** sebagai base model (transfer learning).

### Perbaikan dari `main.ipynb`:
1. **Base model**: EfficientNetV2-B0 (lebih akurat dari MobileNetV2)
2. **Input size**: 224×224 (sesuai pretrained weights, bukan 128)
3. **Preprocessing**: Menggunakan `preprocess_input` bawaan EfficientNetV2 (bukan `rescale=1/255`)
4. **Head arsitektur**: Lebih dalam dengan BatchNorm
5. **Fine-tuning**: Recompile setelah unfreeze
6. **Label smoothing**: Mencegah overconfident predictions
7. **Class weighting**: Menangani imbalanced dataset
8. **Augmentasi**: Ditambah contrast variation

# 1. Environment Setup

In [ ]:
import sys
print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
assert (3, 9) <= sys.version_info < (3, 13), "Python 3.9 - 3.12 diperlukan untuk TensorFlow"

In [ ]:
import os
import time
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

# 2. Konfigurasi

In [ ]:
# === Konfigurasi ===
ROOT_DIR = Path.cwd()

IMG_SIZE       = 224        # EfficientNetV2 pretrained pada 224×224
BATCH_SIZE     = 32
NUM_CLASSES    = 20
EPOCHS_FASE1   = 30        # Fase 1: frozen base
EPOCHS_FASE2   = 30        # Fase 2: fine-tuning
SEED           = 42
SPLIT_RATIO    = (0.70, 0.15, 0.15)

INPUT_DIR   = ROOT_DIR / "dataset_aksara" / "OPSI 1"
OUTPUT_DIR  = ROOT_DIR / "dataset_aksara_split"
MODEL_PATH  = ROOT_DIR / "model_aksara_effnet.keras"
OUTPUT_PLOT = ROOT_DIR / "outputs"

OUTPUT_PLOT.mkdir(exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Model:  {MODEL_PATH}")

# 3. Persiapan Dataset

In [ ]:
# Split dataset (uncomment jika belum di-split)
# import splitfolders
# splitfolders.ratio(
#     str(INPUT_DIR), str(OUTPUT_DIR),
#     seed=SEED, ratio=SPLIT_RATIO,
#     group_prefix=None, move=False
# )

# Verifikasi folder
for split in ['train', 'val', 'test']:
    p = OUTPUT_DIR / split
    assert p.exists(), f"Directory {p} tidak ditemukan!"
    count = sum(len(os.listdir(p / k)) for k in os.listdir(p) if (p / k).is_dir())
    print(f"{split:>5}: {count} images")

In [ ]:
# Verifikasi distribusi kelas
def cek_distribusi(folder):
    kelas = sorted(os.listdir(folder))
    counts = {k: len(os.listdir(os.path.join(folder, k)))
              for k in kelas if os.path.isdir(os.path.join(folder, k))}
    print(f"Distribusi kelas di '{folder.name}':")
    for k, v in counts.items():
        print(f"  {k:<20}: {v:>6}")
    print(f"  Total: {sum(counts.values())} | Min: {min(counts.values())} | Max: {max(counts.values())}")
    return counts

print("=" * 50)
train_counts = cek_distribusi(OUTPUT_DIR / "train")
print("=" * 50)

# 4. Data Loading & Augmentasi

**Penting**: EfficientNetV2 memiliki preprocessing khusus. JANGAN gunakan `rescale=1/255`.
Gunakan `tf.keras.applications.efficientnet_v2.preprocess_input` yang menormalisasi ke range yang benar.

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

# Augmentasi training (TANPA rescale — preprocessing dilakukan oleh preprocess_input)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest',
    horizontal_flip=False          # Aksara tidak boleh di-flip horizontal
)

# Validasi & test: hanya preprocessing, tanpa augmentasi
val_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_set = train_datagen.flow_from_directory(
    str(OUTPUT_DIR / "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED
)

val_set = val_datagen.flow_from_directory(
    str(OUTPUT_DIR / "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_set = test_datagen.flow_from_directory(
    str(OUTPUT_DIR / "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_set.class_indices.keys())
print(f"\nKelas ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"Class indices: {train_set.class_indices}")

# Simpan class_names
with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=4)
print("class_names.json disimpan.")

# 5. Hitung Class Weights

Menangani ketidakseimbangan kelas agar model tidak bias ke kelas mayoritas.

In [ ]:
# Hitung class weights dari distribusi training set
y_train = train_set.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))

print("Class weights:")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name:<20}: {class_weight_dict[idx]:.3f}")

# 6. Pembuatan Model — EfficientNetV2-B0

### Arsitektur head:
```
EfficientNetV2-B0 (frozen) → GlobalAveragePooling2D → BatchNorm 
  → Dense(512, relu) → BatchNorm → Dropout(0.3)
  → Dense(256, relu) → Dropout(0.3)
  → Dense(20, softmax)
```

In [ ]:
def buat_model_efficientnet(num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    """Buat model EfficientNetV2-B0 dengan head yang lebih baik."""
    
    # Load base model tanpa top layers
    base = tf.keras.applications.EfficientNetV2B0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Fase 1: freeze semua layer base
    base.trainable = False
    
    # Head classifier
    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    
    return model, base


def fine_tune_model(model, base_model, unfreeze_dari_layer=150, lr=1e-5):
    """Unfreeze sebagian layer atas untuk fine-tuning.
    
    PENTING: Recompile model setelah mengubah trainable layers
    agar optimizer state di-reset dengan benar.
    """
    base_model.trainable = True
    
    # Freeze layer awal, unfreeze layer akhir
    for layer in base_model.layers[:unfreeze_dari_layer]:
        layer.trainable = False
    
    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    total_count = len(base_model.layers)
    print(f"Unfreezing {trainable_count}/{total_count} layers dari base model")
    
    # RECOMPILE dengan learning rate kecil
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    
    return model

In [ ]:
# Buat model
model, base_model = buat_model_efficientnet()
model.summary()

# 7. Training

### Fase 1: Training classifier head (base frozen)
- Learning rate: 1e-3
- Base model sepenuhnya frozen
- Hanya melatih head layers

### Fase 2: Fine-tuning (sebagian base unfrozen)
- Learning rate: 1e-5 (sangat kecil)
- Unfreeze layer atas dari base model
- Model di-recompile untuk reset optimizer state

In [ ]:
# === Callbacks Fase 1 ===
callbacks_fase1 = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        str(MODEL_PATH),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

# === Callbacks Fase 2 ===
callbacks_fase2 = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        str(MODEL_PATH),
        monitor='val_loss',       # Monitor loss, bukan accuracy, saat fine-tuning
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-8,
        verbose=1
    )
]

In [ ]:
# === Fase 1: Training classifier head ===
print("=" * 60)
print("FASE 1: Training classifier head (base frozen)")
print("=" * 60)

start = time.time()

history1 = model.fit(
    train_set,
    validation_data=val_set,
    epochs=EPOCHS_FASE1,
    callbacks=callbacks_fase1,
    class_weight=class_weight_dict
)

fase1_time = (time.time() - start) / 60
print(f"\nFase 1 selesai dalam {fase1_time:.1f} menit")

In [ ]:
# === Fase 2: Fine-tuning ===
print("=" * 60)
print("FASE 2: Fine-tuning (unfreeze sebagian base model)")
print("=" * 60)

# Load model terbaik dari fase 1
model = tf.keras.models.load_model(str(MODEL_PATH))

# Unfreeze dan recompile
model = fine_tune_model(model, base_model, unfreeze_dari_layer=150, lr=1e-5)

start2 = time.time()

history2 = model.fit(
    train_set,
    validation_data=val_set,
    initial_epoch=len(history1.history['loss']),
    epochs=len(history1.history['loss']) + EPOCHS_FASE2,
    callbacks=callbacks_fase2,
    class_weight=class_weight_dict
)

fase2_time = (time.time() - start2) / 60
total_time = fase1_time + fase2_time
print(f"\nFase 2 selesai dalam {fase2_time:.1f} menit")
print(f"Total training: {total_time:.1f} menit")

# 8. Visualisasi Training

In [ ]:
def plot_history(histories, labels):
    """Plot akurasi dan loss dari semua fase training."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for h, label in zip(histories, labels):
        axes[0].plot(h.history['accuracy'],     label=f'{label} train')
        axes[0].plot(h.history['val_accuracy'],  label=f'{label} val', linestyle='--')
        axes[1].plot(h.history['loss'],          label=f'{label} train')
        axes[1].plot(h.history['val_loss'],       label=f'{label} val', linestyle='--')
    
    axes[0].set_title('Akurasi per Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].set_title('Loss per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_PLOT / 'training_curve_effnet.png'), dpi=150)
    plt.show()

plot_history([history1, history2], ['Fase 1 (Frozen)', 'Fase 2 (Fine-tune)'])

# 9. Evaluasi pada Test Set

In [ ]:
# Load model terbaik
model = tf.keras.models.load_model(str(MODEL_PATH))

# Evaluasi di test set (belum pernah dilihat model)
test_set.reset()
Y_pred = model.predict(test_set, verbose=1)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_set.classes

# Classification report
print("=" * 60)
print("EVALUASI AKHIR — TEST SET")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Test Set (EfficientNetV2-B0)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(str(OUTPUT_PLOT / 'confusion_matrix_effnet.png'), dpi=150)
plt.show()

In [ ]:
# Analisis kelas dengan error terbanyak
errors_per_class = {}
for i, kelas in enumerate(CLASS_NAMES):
    mask  = y_true == i
    total = mask.sum()
    salah = (y_pred[mask] != i).sum()
    errors_per_class[kelas] = {
        'total': int(total),
        'salah': int(salah),
        'error_rate': round(salah / total * 100, 1) if total > 0 else 0
    }

print("\nTop 5 kelas dengan error rate tertinggi:")
sorted_errors = sorted(errors_per_class.items(), key=lambda x: x[1]['error_rate'], reverse=True)
for kelas, info in sorted_errors[:5]:
    print(f"  {kelas}: {info['error_rate']}% salah ({info['salah']}/{info['total']})")

# 10. Test Inferensi OCR

In [ ]:
import cv2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as effnet_preprocess

class OCRPipelineEfficientNet:
    """Pipeline OCR menggunakan model EfficientNetV2-B0.
    
    Penting: Preprocessing harus cocok dengan training.
    Menggunakan preprocess_input dari EfficientNetV2, bukan rescale 1/255.
    """
    def __init__(self, model_path, class_names, img_size=224, confidence_threshold=0.6):
        self.model = tf.keras.models.load_model(model_path)
        self.class_names = class_names
        self.img_size = img_size
        self.conf_thresh = confidence_threshold

    def binarisasi(self, img_bgr):
        """Ubah foto BGR menjadi biner."""
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (7, 7), 0)
        biner = cv2.adaptiveThreshold(
            blur, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 21, 4
        )
        kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        biner = cv2.morphologyEx(biner, cv2.MORPH_CLOSE, kernel_close)
        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        biner = cv2.dilate(biner, kernel_dilate, iterations=1)
        return biner

    def segmentasi_karakter(self, biner):
        """Segmentasi karakter dengan contour + NMS."""
        img_h, img_w = biner.shape[:2]
        img_area = img_h * img_w
        contours, _ = cv2.findContours(biner, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        bounding_box = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            area = w * h
            if area < 500 or area > img_area * 0.25:
                continue
            if w < 15 or h < 15:
                continue
            aspect = w / h
            if aspect < 0.15 or aspect > 6.0:
                continue
            bounding_box.append((x, y, w, h))

        if not bounding_box:
            return []

        boxes_for_nms = [[x, y, w, h] for x, y, w, h in bounding_box]
        scores = [1.0] * len(boxes_for_nms)
        indices = cv2.dnn.NMSBoxes(boxes_for_nms, scores, 0.0, 0.25)
        if len(indices) > 0:
            indices = indices.flatten()
            bounding_box = [bounding_box[i] for i in indices]

        median_h = sorted([b[3] for b in bounding_box])[len(bounding_box) // 2]
        row_tol = max(median_h * 0.6, 20)
        bounding_box.sort(key=lambda b: (b[1] // row_tol, b[0]))
        return bounding_box

    def klasifikasi_batch(self, img_bgr, bboxes):
        """Klasifikasi batch dengan preprocessing EfficientNetV2."""
        if not bboxes:
            return []

        crops = []
        for x, y, w, h in bboxes:
            pad = 4
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2, y2 = min(img_bgr.shape[1], x + w + pad), min(img_bgr.shape[0], y + h + pad)
            crop = img_bgr[y1:y2, x1:x2]
            resized = cv2.resize(crop, (self.img_size, self.img_size))
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            # Gunakan preprocess_input EfficientNetV2 (bukan /255)
            preprocessed = effnet_preprocess(rgb.astype('float32'))
            crops.append(preprocessed)

        batch = np.stack(crops, axis=0)
        all_probs = self.model.predict(batch, verbose=0)

        hasil = []
        for probs, bbox in zip(all_probs, bboxes):
            pred_idx = np.argmax(probs)
            confidence = float(probs[pred_idx])
            hasil.append({
                'kelas': self.class_names[pred_idx],
                'confidence': round(confidence, 3),
                'valid': confidence >= self.conf_thresh,
                'bbox': bbox
            })
        return hasil

    def proses(self, image_path, visualisasi=True):
        """Pipeline lengkap OCR pada satu gambar."""
        img = cv2.imread(str(image_path))
        if img is None:
            raise ValueError(f"Gambar tidak ditemukan: {image_path}")

        biner = self.binarisasi(img)
        boxes = self.segmentasi_karakter(biner)

        if not boxes:
            print("Tidak ada karakter yang terdeteksi.")
            return []

        hasil = self.klasifikasi_batch(img, boxes)

        if visualisasi:
            vis = img.copy()
            for h in hasil:
                x, y, w, hh = h['bbox']
                warna = (0, 200, 0) if h['valid'] else (0, 0, 200)
                cv2.rectangle(vis, (x, y), (x + w, y + hh), warna, 2)
                label = f"{h['kelas']} {h['confidence']:.2f}"
                cv2.putText(vis, label, (x, y - 6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, warna)
            plt.figure(figsize=(14, 8))
            plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.title('Hasil OCR Aksara Jawa (EfficientNetV2-B0)')
            plt.tight_layout()
            plt.show()

        valid = [h for h in hasil if h['valid']]
        print(f"Ditemukan {len(boxes)} karakter | {len(valid)} di atas threshold {self.conf_thresh}")
        print("Prediksi:", " ".join([h['kelas'] for h in valid]))
        return hasil

In [ ]:
# Test OCR pada gambar
ocr = OCRPipelineEfficientNet(
    model_path=str(MODEL_PATH),
    class_names=CLASS_NAMES,
    img_size=IMG_SIZE,
    confidence_threshold=0.6
)

# Ganti path ke gambar test
hasil = ocr.proses('aksara1.png', visualisasi=True)